# 第17章 图像分割与提取（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 17 章跟随代码。**全量收录原书代码（例 17.1～例 17.9）；缺失例题已按书补全**——本章 9 个例题原本齐全，按笔记补全的是 17.1.2 函数语法/参数说明块与 K-means 拓展说明，代码以网页笔记 docs/opencv/ch17.md 为准（含笔误修正与素材适配）。
> 本章绘图用 matplotlib，cv2.imshow / cv_show 需在本地 Python 环境运行（弹窗显示，Notebook 服务内看不到窗口）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg 等素材都在这里），否则 imread 返回 None。
> 素材替换：原书 water_coins.jpg（浅色桌面上的硬币）用 picture13.jpg 顶替——深底亮币，阈值极性改用 THRESH_BINARY、开运算前先闭运算填掉币面字样的洞；lenacolor.png（512×512）用 picture3.jpg（736×1110 人像）顶替，矩形框与模板坐标已按此图重设；my.bmp 用 picture9.jpg 顶替；mask.png 由程序生成（原书用画笔手绘）。代码逻辑除注明处外一字未改，控制台打印的具体数值随素材不同而不同。


## 本章总览

图像分割（image segmentation）= 把前景对象从背景中分离、提取出来。阈值、形态学、轮廓、边缘检测这些"一刀切"的手段在前景连片、背景复杂时不够用，本章给出两套半自动方案，并以拓展形式补上第三种思路。全章例题：例 17.1（形态学获取边界）、例 17.2（距离变换获取确定前景）、例 17.3（确定背景/前景/未知区域）、例 17.4（连通域标注）、例 17.5（标注修正）、例 17.6（分水岭完整分割）、例 17.7～17.9（GrabCut 矩形/掩模/自定义模板）。

- **分水岭算法**：把灰度图当地形，用"开运算去噪 → 膨胀得确定背景 → 距离变换 + 阈值化得确定前景 → 连通域标注 → 修正标注 → watershed"七步流水线喂给算法几颗"种子"，让水漫堤成界；
- **GrabCut 交互式前景提取**：一个矩形框（必要时补几笔掩模）起步，靠高斯混合模型（GMM）+ 图割（graph cut）迭代把前景抠出来；
- **K-means 聚类分割**（拓展）：直接在颜色空间里对像素聚类，类心颜色替换像素，完成一次按颜色的分割。


## 17.1 用分水岭算法实现图像分割与提取

**算法原理（17.1.1）**：分水岭算法把图像形象地比喻为地理学上的地形表面（topographic surface）来实现分割——灰度值高的区域是山峰、灰度值低的区域是山谷。向每一个山谷中"灌注"不同颜色的水，随着水位升高，为防止不同山谷的水交汇，需要在水流可能汇合的地方构建堤坝。这个过程把图像分成两个集合：**集水盆地**（catchment basin）与**分水岭线**（watershed line），构建的堤坝就是分水岭线，也就是对原始图像的分割。

受噪声等因素影响，基础分水岭算法经常得到**过度分割**（over-segmentation）结果：图像被划分成一个个稠密的独立小块，分割失去意义。改进方案是**基于掩模的分水岭算法**（marker-based watershed）：允许用户把认为是同一分割区域的部分标注出来（被标注的部分称为掩模，mask），算法处理时把标注的部分当作同一个分割区域，过度分割明显改善。


### 17.1.2 相关函数介绍

在 OpenCV 中用函数 `cv2.watershed()` 实现分水岭算法；具体实现中还需要借助形态学函数、距离变换函数 `cv2.distanceTransform()` 和连通域标注函数 `cv2.connectedComponents()`。本节逐一练手（例 17.1～例 17.5），最后在例 17.6 拼成完整流水线：

- **形态学函数**：开运算（先腐蚀、后膨胀）去除图像内的噪声；通过形态学操作和减法运算还能**获取图像的边界**——原始图像减去它的腐蚀结果，剩下的一圈就是边界。但形态学操作只适用于比较简单的图像，前景对象存在连接时无法准确获取各子图像的边界；
- **cv2.distanceTransform()**：距离变换（见例 17.2）；
- **cv2.connectedComponents()**：连通域标注（见例 17.4）；
- **cv2.watershed()**：分水岭分割（见 17.1.3）。


### 例 17.1 · 形态学变换获取图像边界

用"原始图像 − 腐蚀结果"获取 picture9.jpg（原书 my.bmp，黑底白色图形）的边界：腐蚀让前景整体向内收缩一圈，二者相减的差集就是一圈边界，比较准确地显示出前景对象的边界。

**素材适配**：picture9.jpg 实为 4 通道 BGRA（扩展名 jpg 的 PNG）——减法会把 alpha 一并清零导致边界显示不出来，代码里先转成 3 通道 BGR 再处理。


In [ ]:
# ===== 17.1.2 例 17.1：形态学 + 减法获取图像边界 =====
import cv2
import numpy as np
import matplotlib.pyplot as plt

o = cv2.imread("picture9.jpg", cv2.IMREAD_UNCHANGED)  # 原书 my.bmp：黑底白色图形
if o.ndim == 3 and o.shape[2] == 4:    # 素材适配：picture9.jpg 实为 4 通道 BGRA（扩展名 jpg 的 PNG），
    o = cv2.cvtColor(o, cv2.COLOR_BGRA2BGR)   # 减法会把 alpha 一并清零致边界显示不出来，先转 3 通道
k = np.ones((5, 5), np.uint8)      # 5×5 全 1 结构元
e = cv2.erode(o, k)                # 腐蚀：前景整体向内收缩一圈
b = cv2.subtract(o, e)             # 原图减腐蚀图：差集就是一圈边界

plt.subplot(131)                   # 1 行 3 列的第 1 幅
plt.imshow(o)                      # 原始图像
plt.axis('off')
plt.subplot(132)
plt.imshow(e)                      # 腐蚀结果
plt.axis('off')
plt.subplot(133)
plt.imshow(b)                      # 边界图像
plt.axis('off')
plt.show()                         # 显示画布


### 例 17.2 · 距离变换获取确定前景

图像内的各个子图连接在一起时，前景就难确定了，此时借助距离变换可以方便地把前景提取出来。`cv2.distanceTransform(src, distanceType, maskSize[, dstType])` 计算二值图像内**非零值像素点到最近的零值像素点的距离**（像素点本身值为 0 则距离也为 0）：前景对象的中心（质心）距离零值像素较远得到较大的值，边缘较近得到较小的值。src 须为 8 位单通道的二值图像；distanceType 常用 `cv2.DIST_L2`（欧氏距离）、`cv2.DIST_L1`（曼哈顿距离）、`cv2.DIST_C`（切比雪夫距离）；maskSize 常用 3/5（DIST_L2 可用 DIST_MASK_PRECISE=0 精确计算）；dstType 默认 CV_32F。

对 picture13.jpg（原书 water_coins.jpg）：Otsu 二值化 → 闭运算填洞 → 开运算×2 去噪后做距离变换，再以 `0.7 * dist_transform.max()` 阈值化，得到 15 个与硬币一一对应的"**确定前景**"（sure foreground）亮斑——通常指前景对象的中心，它们距离背景点足够远。

**关于 0.7 系数**：决定"确定前景"的松紧——取大（接近 1）只留下离边界非常远的点，前景可能碎成多块；取小则相邻物体的确定前景粘连成片，分水岭失去种子。0.7×max 是教程通用经验值，可在 0.5～0.8 之间按粘连/碎块情况微调。


In [ ]:
# ===== 17.1.2 例 17.2：距离变换获取确定前景 =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('picture13.jpg')          # 原书 water_coins.jpg
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)   # 转灰度
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)     # matplotlib 按 RGB 显示，先转序
ishow = img.copy()                         # 留一份原始图用于展示

ret, thresh = cv2.threshold(gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # Otsu 自动阈值。picture13 是深底亮币，用 BINARY；
                                           # 原书 water_coins 是浅底暗币，用 THRESH_BINARY_INV
closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
    np.ones((15, 15), np.uint8))           # 素材适配：闭运算填掉币面深色"5角"字样留下的洞
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN,
    kernel, iterations=2)                  # 开运算×2 去噪（原书对 thresh 直接开运算）

dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)   # 距离变换
ret, fore = cv2.threshold(dist_transform,
    0.7 * dist_transform.max(), 255, 0)    # 以 0.7 倍最大值做阈值化，得到确定前景

plt.subplot(131)                           # 1 行 3 列第 1 幅
plt.imshow(ishow)                          # 原始图像
plt.axis('off')
plt.subplot(132)
plt.imshow(dist_transform)                 # 距离图像
plt.axis('off')
plt.subplot(133)
plt.imshow(fore)                           # 阈值化后的确定前景
plt.axis('off')
plt.show()


### 例 17.3 · 确定背景、确定前景与未知区域

形态学**膨胀**操作把前景"膨胀放大"，前景放大后背景就被"压缩"，此时得到的背景一定小于实际背景——是不包含前景的"**确定背景**"（sure background，记作 B）。有了确定前景 F（例 17.2）和确定背景 B，剩下的区域就是**未知区域**（unknown region）UN——正是分水岭算法要进一步明确的区域：

$$UN = O - B - F \quad\Rightarrow\quad UN = (O - B) - F$$

上式中的"图像 O − 确定背景 B"通过对图像进行形态学的膨胀操作得到。对 picture13.jpg 四联图展示：左上原图 ishow、右上膨胀结果 bg（注意其背景部分才是"确定背景"，前景的一个个小圆是"原始图像 − 确定背景"部分）、左下确定前景 fore、右下由 `cv2.subtract(bg, fore)` 得到的未知区域 un（小圆环）。


In [ ]:
# ===== 17.1.2 例 17.3：确定背景、确定前景、未知区域 =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('picture13.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)     # 转灰度
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)       # 转 RGB 供 matplotlib 显示
ishow = img.copy()                               # 原始图像副本

ret, thresh = cv2.threshold(gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)         # Otsu 阈值分割（深底亮币用 BINARY）
closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
    np.ones((15, 15), np.uint8))                 # 素材适配：填掉币面字样的洞
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN,
    kernel, iterations=2)                        # 开运算去噪

bg = cv2.dilate(opening, kernel, iterations=3)   # 膨胀×3：前景放大，背景被压缩
                                                 # bg 的背景部分 = 确定背景 B
dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)   # 距离变换
ret, fore = cv2.threshold(dist, 0.7 * dist.max(), 255, 0)   # 确定前景 F
fore = np.uint8(fore)                            # 转成 uint8 方便后续运算
un = cv2.subtract(bg, fore)                      # UN = (O - B) - F

plt.subplot(221)                                 # 2 行 2 列第 1 幅
plt.imshow(ishow)                                # 原始图像
plt.axis('off')
plt.subplot(222)
plt.imshow(bg)                                   # 膨胀结果
plt.axis('off')
plt.subplot(223)
plt.imshow(fore)                                 # 确定前景
plt.axis('off')
plt.subplot(224)
plt.imshow(un)                                   # 未知区域
plt.axis('off')
plt.show()


### 例 17.4 · 连通域标注确定前景

明确确定前景后，就可以用 `cv2.connectedComponents()` 对确定前景图像进行标注了：

```text
retval, labels = cv2.connectedComponents(image)
```

image 为 8 位单通道的待标注图像；retval 是返回的标注数量，labels 是标注的结果图像。数值对应关系：**0 代表背景区域，从 1 开始的值代表不同的前景区域**。对例 17.2 的确定前景标注后，15 个硬币中心各得一色（实跑 ret=16，含背景）。


In [ ]:
# ===== 17.1.2 例 17.4：连通域标注确定前景 =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('picture13.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)     # 转灰度
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)       # 转 RGB 显示
ishow = img.copy()                               # 原始图像副本

ret, thresh = cv2.threshold(gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)         # Otsu 阈值分割
closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
    np.ones((15, 15), np.uint8))                 # 素材适配：填洞
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN,
    kernel, iterations=2)                        # 开运算去噪

sure_bg = cv2.dilate(opening, kernel, iterations=3)      # 确定背景（本例未直接使用）
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)   # 距离变换
ret, fore = cv2.threshold(dist_transform,
    0.7 * dist_transform.max(), 255, 0)          # 确定前景
fore = np.uint8(fore)                            # 转 uint8

ret, markers = cv2.connectedComponents(fore)     # 连通域标注：背景 0，对象 1 起
print(ret)                                       # 打印标注数量（含背景）

plt.subplot(131)
plt.imshow(ishow)                                # 原始图像
plt.axis('off')
plt.subplot(132)
plt.imshow(fore)                                 # 确定前景的中心点图像
plt.axis('off')
plt.subplot(133)
plt.imshow(markers)                              # 标注结果
plt.axis('off')
plt.show()


### 例 17.5 · 标注修正，未知区域置 0

分水岭算法约定**标注值 0 代表未知区域**，与 connectedComponents 里"0 是背景"的含义相反，所以标注结果要先修正再喂给 watershed：把结果整体加 1（处理后 1 代表背景、从 2 开始的值代表不同前景区域），再把已经计算出来的未知区域重新标注为 0。关键代码：

```python
ret, markers = cv2.connectedComponents(fore)
markers = markers + 1              # 整体加 1：1 是背景，从 2 起是前景
markers[unknown == 255] = 0        # 未知区域置 0
```

对比图：左图是直接标注的结果；右图是修正后的标注结果，每个确定前景都有一个黑色边缘——这个边缘就是被标注的未知区域。受对比度影响，插图上可能不明显，运行代码在屏幕上观察更清楚。


In [ ]:
# ===== 17.1.2 例 17.5：标注修正，未知区域置 0 =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('picture13.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)     # 转灰度
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)       # 转 RGB 显示
ishow = img.copy()

ret, thresh = cv2.threshold(gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)         # Otsu 阈值分割
closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
    np.ones((15, 15), np.uint8))                 # 素材适配：填洞
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN,
    kernel, iterations=2)                        # 开运算去噪
sure_bg = cv2.dilate(opening, kernel, iterations=3)      # 确定背景
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
ret, fore = cv2.threshold(dist_transform,
    0.7 * dist_transform.max(), 255, 0)          # 确定前景
fore = np.uint8(fore)

ret, markers1 = cv2.connectedComponents(fore)    # 直接标注的结果
foreAdv = fore.copy()                            # 复制一份确定前景
unknown = cv2.subtract(sure_bg, foreAdv)         # 未知区域 = 确定背景 - 确定前景
ret, markers2 = cv2.connectedComponents(foreAdv) # 重新标注
markers2 = markers2 + 1                          # 整体加 1：1 是背景，2 起是前景
markers2[unknown == 255] = 0                     # 未知区域置 0

plt.subplot(121)                                 # 1 行 2 列第 1 幅
plt.imshow(markers1)                             # 修正前
plt.axis('off')
plt.subplot(122)
plt.imshow(markers2)                             # 修正后
plt.axis('off')
plt.show()


### 17.1.3 分水岭算法图像分割实例

结合前面的知识，用分水岭算法做图像分割，基本步骤为：

1. 通过形态学开运算对原始图像 O 去噪；
2. 通过**膨胀**操作获取"原始图像 − 确定背景"（确定背景 B）；
3. 利用 `cv2.distanceTransform()` 进行运算并阈值处理，得到"确定前景 F"；
4. 计算未知区域 UN（UN = O − B − F）；
5. 利用 `cv2.connectedComponents()` 标注；
6. 对标注结果进行修正（整体 +1、未知区域置 0）；
7. 使用分水岭函数完成分割。

**原书笔误**：步骤 2 原书写作"通过腐蚀操作获取确定背景"，与例 17.3、例 17.6 的代码 `cv2.dilate(opening, ...)` 不符——确定背景是**膨胀**结果剩下的背景部分。

OpenCV 中实现分水岭算法的函数是 `cv2.watershed(image, markers)`：image 必须是 8 位三通道图像；markers 是 32 位单通道标注图，使用前先用正数大致勾画出期望分割的区域（每个分割区域标注为 1、2、3 等，尚未确定的区域标注为 0），可理解为分水岭分割的"种子"区域。**两个参数都必填**，原书"markers 可以省略"的说法有误——markers 正是算法赖以工作的种子输入。返回后**边界像素被写成 -1**（包括不同分割区域交界处与图像最外圈一像素框），用 `img[markers == -1] = [0, 255, 0]` 涂色即可可视化分割线。

### 例 17.6 · 分水岭算法完整分割流程

对 picture13.jpg 走完七步，15 枚硬币各自成区、绿色为分割线。素材适配两处：picture13 是深底亮币，Otsu 用 `THRESH_BINARY`（原书 water_coins 浅底暗币用 BINARY_INV，极性用反会把背景当成前景）；币面深色"5角"字样会在前景掩模里留洞、干扰距离变换，开运算之前先用 15×15 结构元闭运算填洞。


In [ ]:
# ===== 17.1.3 例 17.6：分水岭算法完整分割流程 =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('picture13.jpg')                # 原书 water_coins.jpg
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)     # 转灰度
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)       # 转 RGB 显示
ishow = img.copy()                               # 留一份原始图

ret, thresh = cv2.threshold(gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU)         # 步骤 0：Otsu 二值化（深底亮币用 BINARY）
closing = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE,
    np.ones((15, 15), np.uint8))                 # 素材适配：闭运算填掉币面深色字样的洞
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(closing, cv2.MORPH_OPEN,
    kernel, iterations=2)                        # 步骤 1：开运算去噪

sure_bg = cv2.dilate(opening, kernel, iterations=3)       # 步骤 2：膨胀得到确定背景 B
dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
ret, sure_fg = cv2.threshold(dist_transform,
    0.7 * dist_transform.max(), 255, 0)          # 步骤 3：距离变换 + 阈值化得到确定前景 F
sure_fg = np.uint8(sure_fg)                      # 转 uint8
unknown = cv2.subtract(sure_bg, sure_fg)         # 步骤 4：未知区域 UN = (O - B) - F

ret, markers = cv2.connectedComponents(sure_fg)  # 步骤 5：对确定前景标注
markers = markers + 1                            # 步骤 6a：整体加 1，让背景=1、前景从 2 起
markers[unknown == 255] = 0                      # 步骤 6b：未知区域置 0

markers = cv2.watershed(img, markers)            # 步骤 7：分水岭分割，边界写 -1
img[markers == -1] = [0, 255, 0]                 # 把边界涂成绿色

plt.subplot(121)                                 # 1 行 2 列第 1 幅
plt.imshow(ishow)                                # 原始图像
plt.axis('off')
plt.subplot(122)
plt.imshow(img)                                  # 分割结果：绿色为分割线
plt.axis('off')
plt.show()


## 17.2 交互式前景提取

2004 年，微软研究院（剑桥）的 Rother 等人在论文 *GrabCut: Interactive Foreground Extraction Using Iterated Graph Cuts* 中提出了**交互式前景提取**（interactive foreground extraction）技术：仅需要很少的交互操作，就能准确地提取出前景图像。先用矩形框指定前景所在的大致位置范围并不断迭代分割；效果不理想时由用户干预——用白色标注要提取为前景的区域、黑色标注要作为背景的区域，把标注后的图像作为掩模继续迭代，直到得到最终结果。

算法流程：矩形框内是既含前景又含背景的**未确定区域**、框外是"确定背景"→ 用高斯混合模型（GMM）对前景和背景分别建模，对未分类像素按其与已知分类像素的关系分类 → 根据像素分布生成一幅图（graph）：每个像素是一个节点，另有两个节点——前景节点和背景节点，边权由像素属于前景/背景的概率及相邻像素的相似性（颜色越接近权值越大）决定 → 在连通图上根据边权做图割，把点划分为前景节点和背景节点 → 不断重复直至分类收敛。

```text
mask, bgdModel, fgdModel = cv2.grabCut(img, mask, rect, bgdModel, fgdModel, iterCount[, mode])
```

| 参数/返回值 | 含义 |
| --- | --- |
| img | 输入图像，8 位 3 通道 |
| mask | 掩模图像，8 位单通道，只能取 4 个值：0（cv2.GC_BGD 确定背景）、1（cv2.GC_FGD 确定前景）、2（cv2.GC_PR_BGD 可能的背景）、3（cv2.GC_PR_FGD 可能的前景）；提取时把 0 和 2 合并为背景、1 和 3 合并为前景 |
| rect | 前景大致范围 (x, y, w, h)。区域外的部分被认为是"确定背景"，务必让前景完整落在 rect 内；使用掩模模式时设为 None |
| bgdModel / fgdModel | 算法内部使用的数组，创建大小为 (1, 65) 的 numpy.float64 数组即可 |
| iterCount | 迭代次数 |
| mode | cv2.GC_INIT_WITH_RECT 使用矩形模板；cv2.GC_INIT_WITH_MASK 使用自定义模板（二者能组合使用，ROI 外的像素自动处理为背景）；cv2.GC_EVAL 修复模式；cv2.GC_EVAL_FREEZE_MODEL 固定模式 |

**原书笔误**：原书写"通过转换将其中的白色像素设置为 0，黑色像素设置为 1"，方向反了——正确对应关系是白色（255）映射为 1（确定前景）、黑色（0）映射为 0（确定背景），与例 17.8 的代码一致。


### 例 17.7 · 矩形模式 GrabCut

只用矩形 rect=(60, 100, 620, 850)（原书 lena 用 (50, 50, 400, 500)）对 picture3.jpg 做初步提取：掩模全 0 时 `cv2.grabCut()` 的处理效果并不太好——picture3 中人物深色的头发、眼睛、嘴和衣服都被当成了背景，只有颜色与背景差异明显的面部皮肤被提取出来。用 `np.where((mask == 2) | (mask == 0), 0, 1)` 把四值掩模并为二值后与原图相乘完成提取。这正是需要掩模标注改进的原因。


In [ ]:
# ===== 17.2 例 17.7：矩形模式 GrabCut =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

o = cv2.imread('picture3.jpg')                   # 原书 lenacolor.png（512×512）
orgb = cv2.cvtColor(o, cv2.COLOR_BGR2RGB)        # 转 RGB 显示
mask = np.zeros(o.shape[:2], np.uint8)           # 全 0 掩模：初始全部当作确定背景
bgdModel = np.zeros((1, 65), np.float64)         # 算法内部数组
fgdModel = np.zeros((1, 65), np.float64)
rect = (60, 100, 620, 850)                       # 前景大致范围 (x,y,w,h)；原书 lena 用 (50,50,400,500)
cv2.grabCut(o, mask, rect, bgdModel, fgdModel, 5,
    cv2.GC_INIT_WITH_RECT)                       # 矩形模式，迭代 5 次

mask2 = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')   # 0/2 当背景，1/3 当前景
ogc = o * mask2[:, :, np.newaxis]                # 掩模逐通道乘原图：背景清零
ogc = cv2.cvtColor(ogc, cv2.COLOR_BGR2RGB)       # 转 RGB 显示

plt.subplot(121)                                 # 1 行 2 列第 1 幅
plt.imshow(orgb)                                 # 原始图像
plt.axis('off')
plt.subplot(122)
plt.imshow(ogc)                                  # GrabCut 提取结果
plt.axis('off')
plt.show()


### 例 17.8 · 掩模模式 GrabCut

先矩形模式初步提取，再用手绘模板标注细化。需要注意：画笔标记的模板图像 m0 **不能**直接作为 mask 参数使用——`cv2.grabCut()` 要求 mask 的值必须是 0、1、2、3 之一，而 m0 中存在 [0, 255] 内的值，必须先把 m0 的白色（255）映射为 1（确定前景）、黑色（0）映射为 0（确定背景）。原书用 Windows 画笔在图上手工涂抹并另存 mask.png；这里用程序生成等价的"手绘"模板（脸部涂白保留、两个背景角涂黑删除、其余涂中性灰不干预）并存成 mask.png，与原书流程对齐。补充白色标注后，发顶、嘴等区域被找了回来——但模板没盖到的右眼仍被误删：标注要覆盖全部前景细节，提取效果才完整。


In [ ]:
# ===== 17.2 例 17.8：掩模模式 GrabCut =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

o = cv2.imread('picture3.jpg')                   # 原书 lenacolor.png
orgb = cv2.cvtColor(o, cv2.COLOR_BGR2RGB)        # 转 RGB 显示
mask = np.zeros(o.shape[:2], np.uint8)           # 全 0 掩模
bgd = np.zeros((1, 65), np.float64)
fgd = np.zeros((1, 65), np.float64)
rect = (60, 100, 620, 850)                       # 原书 lena 用 (50,50,400,500)
cv2.grabCut(o, mask, rect, bgd, fgd, 5,
    cv2.GC_INIT_WITH_RECT)                       # 步骤 1：矩形模式初步提取

# 原书用 Windows 画笔在图上手工涂抹并另存为 mask.png：
# 白色 = 要保留的前景，黑色 = 要删除的背景。这里用程序生成等价的"手绘"模板：
# 脸部涂白（保留），左上、右上两个背景角涂黑（删除），其余涂中性灰表示不干预
m0 = np.full(o.shape[:2], 127, np.uint8)         # 中性灰 = 不打标记
cv2.rectangle(m0, (220, 320), (520, 720), 255, -1)   # 白笔刷：脸
cv2.rectangle(m0, (0, 0), (120, 160), 0, -1)         # 黑笔刷：左上背景
cv2.rectangle(m0, (616, 0), (736, 200), 0, -1)       # 黑笔刷：右上背景
cv2.imwrite('mask.png', m0)                      # 存成 mask.png，与原书流程对齐

mask2 = cv2.imread('mask.png', 0)                # 以灰度方式读入模板
mask2Show = cv2.imread('mask.png', -1)           # 原样读入用于显示
plt.subplot(121)                                 # 1 行 2 列第 1 幅
plt.imshow(mask2Show, cmap='gray')               # 模板（原书把模板转 RGB 显示）
plt.axis('off')

mask[mask2 == 0] = 0                             # 黑色（0）→ 确定背景
mask[mask2 == 255] = 1                           # 白色（255）→ 确定前景
mask, bgd, fgd = cv2.grabCut(o, mask, None, bgd, fgd, 5,
    cv2.GC_INIT_WITH_MASK)                       # 步骤 7：掩模模式继续迭代
mask = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')   # 合并四值为二值
ogc = o * mask[:, :, np.newaxis]                 # 背景清零
ogc = cv2.cvtColor(ogc, cv2.COLOR_BGR2RGB)

plt.subplot(122)
plt.imshow(ogc)                                  # 提取结果
plt.axis('off')
plt.show()


### 例 17.9 · 直接自定义模板 GrabCut

实际使用中也可以不用矩形初始化，直接使用模板模式：像素值 0 标注确定背景、1 标注确定前景、2 标注可能的背景、3 标注可能的前景。自定义模板的步骤：先用 `numpy.zeros` 构造一幅内部像素值都是 0（确定背景）的图像，再用切片赋值逐步细化——`mask2[100:950, 60:680] = 3` 划分可能的前景（原书 lena 用 `mask2[30:512, 50:400] = 3`），`mask2[320:720, 220:520] = 1` 划分确定前景（不设置这块，头像提取不完整；原书用 `mask2[50:300, 150:200] = 1`）。构造更有针对性的模板，能够提取更准确的前景对象；对于不同的图像，要构造不同的模板来划分它们的确定前景、确定背景、可能的前景与可能的背景。


In [ ]:
# ===== 17.2 例 17.9：直接自定义模板 GrabCut =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

o = cv2.imread('picture3.jpg')                   # 原书 lenacolor.png
orgb = cv2.cvtColor(o, cv2.COLOR_BGR2RGB)        # 转 RGB 显示
bgd = np.zeros((1, 65), np.float64)
fgd = np.zeros((1, 65), np.float64)
mask2 = np.zeros(o.shape[:2], np.uint8)          # 先全部构造为 0（确定背景），后续逐步细化
mask2[100:950, 60:680] = 3                       # 人像的可能前景区域；原书 lena 用 mask2[30:512,50:400]=3
mask2[320:720, 220:520] = 1                      # 人像的确定前景区域；不设置这块，头像提取不完整
                                                 # 原书 lena 用 mask2[50:300,150:200]=1
cv2.grabCut(o, mask2, None, bgd, fgd, 5,
    cv2.GC_INIT_WITH_MASK)                       # 直接掩模模式，不做矩形初始化
mask2 = np.where((mask2 == 2) | (mask2 == 0), 0, 1).astype('uint8')   # 合并四值为二值
ogc = o * mask2[:, :, np.newaxis]                # 背景清零
ogc = cv2.cvtColor(ogc, cv2.COLOR_BGR2RGB)

plt.subplot(121)
plt.imshow(orgb)                                 # 原始图像
plt.axis('off')
plt.subplot(122)
plt.imshow(ogc)                                  # 提取结果
plt.axis('off')
plt.show()


## 拓展：K-means 聚类分割

本节为补充内容，原书在第 22 章系统讲解 K 均值聚类（K-means clustering）。放在这里，是因为 K-means 也是经典的图像分割手段：分水岭靠"地形 + 种子"，GrabCut 靠"交互 + 图割"，K-means 则直接在颜色空间里对像素分类——一幅彩色图的每个像素是 BGR 颜色空间中的一个三维点，把整幅图的像素当作样本集聚成 K 簇，再把每个像素替换为其类心的颜色，即颜色量化（color quantization）：颜色相近的像素归为一类，同色区域自然连成块。

```text
retval, bestLabels, centers = cv2.kmeans(data, K, bestLabels, criteria, attempts, flags)
```

| 参数/返回值 | 含义 |
| --- | --- |
| data | 待处理数据集合，必须 np.float32 类型，每个特征单独一列（图像先摊平成 N×3） |
| K | 要分出的簇的个数 |
| bestLabels | 调用时传 None；返回各数据点的最终分类标签（索引），N×1、int32 |
| criteria | 迭代终止条件 (type, max_iter, eps)：type 取 cv2.TERM_CRITERIA_EPS（精度满足即停）、cv2.TERM_CRITERIA_MAX_ITER（超过迭代数即停）或二者相加（任一满足即停） |
| attempts | 用不同的初始分类值尝试的次数，取结果最好的那次 |
| flags | 初始中心选取方法：cv2.KMEANS_RANDOM_CENTERS 随机 / cv2.KMEANS_PP_CENTERS 中心化 / cv2.KMEANS_USE_INITIAL_LABELS 用户输入 |
| retval | 距离值（紧密度，compactness）：每个点到相应中心点距离的平方和，越小说明簇越紧凑 |
| centers | 每个分类的中心点数据，K×特征数、float32 |


### cv2.kmeans 已知分组对照

先用一组已知分组的二维点验证 `cv2.kmeans()` 的参数与三个返回值：三组各 60 个点（真值中心 (0, 0)、(60, 60)、(120, 0)，固定随机种子可复现），K=3、尝试 10 次聚类。簇编号与真实分组可能整体置换，先按类心就近对齐再对照——实跑三个类心与真值的偏差都在 0.4 以内，180 个点的分组与已知分组完全一致（一致率 1.0）。


In [ ]:
# ===== 拓展：cv2.kmeans 已知分组对照 =====
import numpy as np
import cv2

# 第 1 步：构造三组已知分组的二维点（真值：中心 (0,0)、(60,60)、(120,0)）
rng = np.random.RandomState(42)                    # 固定随机种子，结果可复现
g0 = rng.normal(loc=(0, 0), scale=2.0, size=(60, 2))     # 第 0 组 60 个点
g1 = rng.normal(loc=(60, 60), scale=2.0, size=(60, 2))   # 第 1 组 60 个点
g2 = rng.normal(loc=(120, 0), scale=2.0, size=(60, 2))   # 第 2 组 60 个点
true_labels = np.array([0] * 60 + [1] * 60 + [2] * 60)   # 已知分组标签
data = np.vstack([g0, g1, g2]).astype(np.float32)        # 合并并转 float32（cv2.kmeans 硬性要求）

# 第 2 步：设置参数并调用
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)   # 迭代 10 次或精度 1.0 即停
compactness, labels, centers = cv2.kmeans(data, 3, None, criteria, 10,
    cv2.KMEANS_RANDOM_CENTERS)                     # K=3，尝试 10 次，随机初始中心

# 第 3 步：对照已知分组（簇编号与真实分组可能整体置换，先按类心就近对齐）
true_centers = np.array([[0, 0], [60, 60], [120, 0]], np.float32)
lab2true = {k: int(np.argmin(np.linalg.norm(true_centers - centers[k], axis=1)))
            for k in range(3)}                     # 每个簇的实际中心 → 最近的已知中心
pred = np.array([lab2true[l] for l in labels.ravel()])   # 簇编号翻译回真实分组
print("compactness =", compactness)                # 紧密度（正浮点数）
print("labels.shape =", labels.shape, "dtype =", labels.dtype)   # (180, 1) int32
print("centers =", np.round(centers, 1))           # 三个类心
print("与已知分组一致率 =", (pred == true_labels).mean())   # 应为 1.0


### K-means 颜色量化分割

把同样的思路用在真实图像上：picture13.jpg 的像素摊平成 N×3 后聚成 4 类（K=4），每个像素替换为其类心颜色再还原成图像形状——实跑 4 个类心分别对应深色皮革背景、两种深浅的金色币面和暗色斑渍，同色区域连成块，完成一次按颜色的粗分割。

**三种分割方法怎么选**：K-means 只看颜色、不看位置，没有空间邻接约束（两块颜色相同的分离区域会被并成一类），适合颜色分明的场景和颜色量化预处理；分水岭种子驱动、边界贴合物体轮廓，能分出相邻同色物体，代价是要搭预处理流水线；GrabCut 交互最少，一个矩形框就能出结果，适合单主体抠图，完全自动时效果不稳，通常要补几笔掩模。


In [ ]:
# ===== 拓展：K-means 颜色量化分割 =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('picture13.jpg')                  # 读入彩色图
Z = img.reshape((-1, 3)).astype(np.float32)        # R×C×3 摊平成 N×3，每行一个像素
K = 4                                              # 聚成 4 种颜色
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
ret, label, center = cv2.kmeans(Z, K, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

center = np.uint8(center)                          # 类心还原成 uint8 颜色
res = center[label.ravel()]                        # 每个像素替换为其类心颜色
quant = res.reshape(img.shape)                     # 还原成图像形状

plt.subplot(121)                                   # 1 行 2 列第 1 幅
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))   # 原图
plt.axis('off')
plt.subplot(122)
plt.imshow(cv2.cvtColor(quant, cv2.COLOR_BGR2RGB)) # 4 色量化结果
plt.axis('off')
plt.show()
print("compactness =", ret)                        # 本次量化的紧密度


## 坑与陷阱（速查）

1. **阈值极性想当然**：`THRESH_BINARY_INV+OTSU` 只对"浅底暗物"成立（原书 water_coins）；深底亮物（picture13）用 INV 会把背景变成前景，后面距离变换、标注、分水岭全在分割背景——先想清楚"物体亮还是背景亮"；
2. **0 在两个函数里含义相反**：connectedComponents 里 0 = 背景，watershed 里 0 = 未知区域——必须 `markers + 1` 后再把 unknown 区域置 0，否则背景被当成 0 号种子、结果整体错位；
3. **markers 的 dtype 必须是 32 位有符号整数**：connectedComponents 的返回值天然是 int32 可直接用；自己构造标注要写 `np.int32`，传 uint8 会报错；
4. **-1 边界比想象的多一圈外框**：`markers == -1` 除了不同区域交界处，还总是包含图像最外圈的一像素框（OpenCV 把图像外侧当虚拟区域参与泛洪），涂绿后结果自带一圈绿框；
5. **确定背景必须用膨胀**：对前景掩模做膨胀（如 iterations=3），前景放大、背景缩小，剩下的背景才"确定不含前景"；直接拿二值化结果当确定背景，UN 算错、边界画不准；
6. **手绘模板不能直接当 mask 传给 grabCut**：画笔画出的是 0～255 的灰度图，grabCut 只认 0/1/2/3——必须先映射白 255→1（确定前景）、黑 0→0（确定背景），再以 GC_INIT_WITH_MASK 模式调用；
7. **rect 必须框住全部前景**：rect 之外一律被当作确定背景，框外的前景永远不会被提取；掩模模式 rect 传 None；
8. **前景里的暗色字样会留洞**：币面"5角"字样、图案比 Otsu 阈值暗时会在前景掩模里挖出黑洞，距离变换的峰值被切碎——开运算前先用较大结构元（如 15×15）闭运算填洞。
